In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 2 — A ANATOMIA DA DIFERENÇA: ENTENDENDO O PROBLEMA DE CLASSIFICAÇÃO

> "Acurácia é um farol em um dia de sol. Útil, mas não revela os perigos que se escondem na névoa. Para navegar na incerteza, precisamos de um mapa mais detalhado da verdade."
> — Notas do diário do Protagonista

No rescaldo do fracasso do **OncoClassify 1.0**, passei semanas obcecado por uma única métrica: **acurácia**. O modelo tinha 97% nos dados de validação. Um número que apresentei com orgulho aos diretores do hospital. Um número que se revelou uma mentira perigosa. A acurácia, sozinha, não me contava a história toda — era como o resumo de um livro complexo: continha a essência, mas omitia os detalhes que mudavam o significado da trama.

Helena foi um desses "detalhes". O modelo estava certo em 97% das vezes, mas no caso dela falhou de forma catastrófica. Ele era um especialista em identificar casos benignos, a maioria, e um amador em detectar a minoria maligna. E essa falha não aparecia no número brilhante de "97%".

Foi então que redescobri a beleza da **Matriz de Confusão**. Não era apenas uma tabela: era um mapa. Um mapa da verdade e da mentira do meu modelo. Ela me mostraria ***quantas vezes*** o modelo errou e, sobretudo, ***como*** ele errou. E no diagnóstico de câncer, o "como" é tudo.

## A Matriz de Confusão: O Mapa da Verdade do Modelo

Se a acurácia é um resumo, a Matriz de Confusão é o relatório completo. Para classificação binária, é uma tabela 2×2 que cruza os valores reais com as previsões, detalhando acertos e erros para cada classe. Ela é a base de praticamente todas as outras métricas.

Lembrando nossa convenção: a classe **positiva** é a **Maligna (0)** — a que queremos detectar.

**Verdadeiro Positivo (VP):** previu Maligno, era Maligno. *Acerto do caso perigoso.*

**Verdadeiro Negativo (VN):** previu Benigno, era Benigno. *Acerto do caso seguro.*

**Falso Positivo (FP):** previu Maligno, era Benigno. *Alarme falso.*

**Falso Negativo (FN):** previu Benigno, era Maligno. *A ameaça silenciosa — o caso de Helena.*

| | Previsto: Maligno (0) | Previsto: Benigno (1) |
|---|---|---|
| **Real: Maligno (0)** | Verdadeiro Positivo (VP) | Falso Negativo (FN) |
| **Real: Benigno (1)** | Falso Positivo (FP) | Verdadeiro Negativo (VN) |

> **⚠️ ATENÇÃO — O Paradoxo da Acurácia**
>
> Em datasets desbalanceados, a acurácia engana. Como vimos no Capítulo 1, um modelo que sempre prevê a classe majoritária (Benigno) teria **63,3%** de acurácia sem nunca identificar um único câncer. Um modelo pode ter acurácia alta e ser completamente inútil. Por isso, nunca avalie um classificador apenas pela acurácia.

## Construindo um Modelo de Base

Para desenhar nossa primeira Matriz de Confusão precisamos de previsões. Ainda não treinamos nada sofisticado, então usamos um **modelo de base (baseline)** ingênuo: prever sempre a classe majoritária (Benigno). Ele estabelece o piso de desempenho e expõe, de forma crua, o quanto a acurácia pode enganar.

#### Código 2.1: Matriz de Confusão do Modelo de Base


In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import confusion_matrix, accuracy_score
from oncoclassify_utils import y   # alvo canonico: 0=Maligno, 1=Benigno

# Baseline ingenuo: prever SEMPRE a classe majoritaria (Benigno = 1).
y_pred_base = np.ones_like(y)

acc = accuracy_score(y, y_pred_base)
cm = confusion_matrix(y, y_pred_base, labels=[0, 1])   # ordem: [Maligno, Benigno]
cm_df = pd.DataFrame(cm, index=["Real: Maligno", "Real: Benigno"],
                         columns=["Previsto: Maligno", "Previsto: Benigno"])

print(f"Acuracia do Modelo de Base: {acc:.4f}\n")
print(cm_df)

Acuracia do Modelo de Base: 0.6333

               Previsto: Maligno  Previsto: Benigno
Real: Maligno                  0                220
Real: Benigno                  0                380


A acurácia é de **63,33%** — exatamente a proporção de benignos. À primeira vista, acertar quase dois terços dos casos não parece terrível. A Matriz de Confusão conta a verdade. Vamos visualizá-la.

#### Código 2.2: A Matriz em Forma de Mapa de Calor


In [ ]:
import seaborn as sns, matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

fig, ax = plt.subplots(figsize=(8, 5.5))
cores = ListedColormap(["#ffd6d6", "#f4a3a3", "#9fd4a0", "#4CAF50"])
sns.heatmap(cm_df, annot=True, fmt="d", cmap=cores, cbar=False, ax=ax,
            annot_kws={"size": 14, "weight": "bold"})
ax.set_title("Matriz de Confusão — Modelo de Base (Sempre Benigno)")
ax.set_ylabel("Verdade Real"); ax.set_xlabel("Previsão do Modelo")
plt.tight_layout(); plt.show()

![Matriz de confusão do baseline](media/figura_02_1.png)

O mapa expõe o desastre por trás da acurácia "razoável":

**VN = 380** — acertou todos os benignos (era sua única previsão).

**FN = 220** — errou **todos** os 220 casos de câncer, classificando-os como benignos. Cada célula aqui é uma Helena em potencial.

**VP = 0** — nunca identificou um único câncer.

**FP = 0** — como nunca previu "Maligno", não gerou alarmes falsos.

Um modelo com 63% de acurácia e **zero** capacidade de detectar a doença que deveria detectar. A acurácia sozinha escondia isso; a Matriz de Confusão revela de imediato.

> **💡 DICA — Sempre crie um modelo de base**
>
> Antes de qualquer modelo complexo, construa um baseline simples (prever a maioria, ou prever ao acaso). Ele fornece o piso de desempenho. Se o seu modelo sofisticado não superar claramente o baseline, algo está errado. É um teste de sanidade essencial.

## Métricas Que Realmente Importam

Da Matriz de Confusão derivam métricas muito mais informativas que a acurácia. Duas são centrais:

**Recall (Sensibilidade)** = VP / (VP + FN) — *"De todos os que eram realmente malignos, quantos o modelo encontrou?"* É a métrica que evita casos como o de Helena; queremos um Recall altíssimo.

**Precisão** = VP / (VP + FP) — *"Quando o modelo diz 'Maligno', com que frequência acerta?"* Baixa precisão significa muitos alarmes falsos, levando a biópsias desnecessárias e ansiedade.

Para o nosso baseline, o **Recall é 0%** (0 / 220) e a **Precisão é indefinida** (0 / 0). O fracasso é completo — e agora temos a linguagem para descrevê-lo com exatidão.

> **📌 CHECKLIST DO CAPÍTULO 2**
>
> ✅ Sabe definir os quatro quadrantes: **VP, VN, FP, FN**.
>
> ✅ Entende o **Paradoxo da Acurácia** em dados desbalanceados.
>
> ✅ Executou o Código 2.1 e viu como um modelo com **63,3%** de acurácia pode ter **0%** de acerto na classe que mais importa.
>
> ✅ Consegue definir **Recall** e **Precisão** e a pergunta que cada um responde.
>
> ✅ Compreende a diferença crítica entre **Falso Positivo** e **Falso Negativo** no contexto médico.
>
> **⚠️ CRÍTICO:** A Matriz de Confusão não é só avaliação, é diagnóstico do modelo. Ela revela o *tipo* de erro cometido, permitindo decisões informadas sobre como melhorá-lo.

A matriz do modelo de base estava na minha tela, uma acusação silenciosa. Ali, na célula [Real: Maligno, Previsto: Benigno], o número **220**. Duzentas e vinte Helenas em potencial. A acurácia de 63% parecia agora uma piada de mau gosto.

Eu finalmente tinha a linguagem para descrever a falha: o problema não era apenas o erro, mas o *tipo* de erro. E entendi as duas forças opostas que precisaria equilibrar: encontrar todos os cânceres (maximizar o Recall) e não alarmar pacientes saudáveis (maximizar a Precisão). Melhorar uma quase sempre piora a outra. Esse seria o meu dilema — o *trade-off* inevitável no coração da classificação médica.
